Partie 1 – Exploration du dataset 
Développer un programme Python capable de récupérer, pour chaque image, son nom, sa classe, 
son format, son mode, sa largeur, sa hauteur, l’écart-type de ses pixels, son nombre de canaux et sa 
taille. 
NB : prendre en charge aussi les fichiers corrompus

In [2]:
# Imports necessaires pour l'atelier de preparation de donnees images
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
import hashlib
import imagehash
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
CATEGORIES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
RAW_DIR = "../data/raw"
CLEANED_DIR = "../data/cleaned"
REPORTS_DIR = "../reports"

In [3]:
# Partie 1 : fonction qui extrait les metadonnees d'une seule image
def extraire_metadonnees(chemin_fichier, classe):
    """
    Recupere les metadonnees d'une image : nom, classe, format, mode,
    largeur, hauteur, ecart-type des pixels, nombre de canaux, taille (octets).
    Gere aussi le cas d'un fichier corrompu (retourne des valeurs NaN + un flag).
    """
    nom_fichier = os.path.basename(chemin_fichier)
    taille_octets = os.path.getsize(chemin_fichier)

    try:
        with Image.open(chemin_fichier) as img:
            img.verify()  # verifie l'integrite du fichier sans le charger entierement
        # On rouvre l'image car verify() rend l'objet inutilisable ensuite
        with Image.open(chemin_fichier) as img:
            img_array = np.array(img)
            format_img = img.format
            mode_img = img.mode
            largeur, hauteur = img.size
            nb_canaux = 1 if img_array.ndim == 2 else img_array.shape[2]
            ecart_type = img_array.std()
            corrompue = False
    except Exception:
        format_img = mode_img = np.nan
        largeur = hauteur = nb_canaux = ecart_type = np.nan
        corrompue = True

    return {
        "nom_fichier": nom_fichier,
        "classe": classe,
        "format": format_img,
        "mode": mode_img,
        "largeur": largeur,
        "hauteur": hauteur,
        "ecart_type_pixels": ecart_type,
        "nb_canaux": nb_canaux,
        "taille_octets": taille_octets,
        "corrompue": corrompue,
        "chemin": chemin_fichier,
    }

In [4]:
# Partie 1 (suite) : parcourir toutes les images de data/raw et construire le tableau d'audit
liste_metadonnees = []

for classe in CATEGORIES:
    dossier_classe = os.path.join(RAW_DIR, classe)
    for nom_fichier in os.listdir(dossier_classe):
        chemin_complet = os.path.join(dossier_classe, nom_fichier)
        if os.path.isfile(chemin_complet):
            meta = extraire_metadonnees(chemin_complet, classe)
            liste_metadonnees.append(meta)

df_audit = pd.DataFrame(liste_metadonnees)
print("Nombre total d'images analysees :", df_audit.shape[0])
df_audit.head()

Nombre total d'images analysees : 1032


,nom_fichier,classe,format,mode,largeur,hauteur,ecart_type_pixels,nb_canaux,taille_octets,corrompue,chemin
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False,../data/raw\cardboard\cardboard1.jpg
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False,../data/raw\cardboard\cardboard10.jpg
2,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,46.108305,3.0,14884,False,../data/raw\cardboard\cardboard100.jpg
3,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,72.263996,3.0,14289,False,../data/raw\cardboard\cardboard101.jpg
4,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,48.388937,3.0,18015,False,../data/raw\cardboard\cardboard102.jpg


Partie 2 – Détecter les images corrompues 
Écrire et se servir d’une fonction qui détecte une image corrompue.

In [5]:
# Partie 2 : fonction qui detecte si une image est corrompue
def est_corrompue(chemin_fichier):
    """
    Retourne True si le fichier n'est pas une image valide (corrompu, tronque,
    ou pas une vraie image malgre son extension), False sinon.
    """
    try:
        with Image.open(chemin_fichier) as img:
            img.verify()
        return False
    except Exception:
        return True

In [6]:
# Utilisation de la fonction sur tout le dataset, a partir de df_audit
images_corrompues = df_audit[df_audit["corrompue"] == True]
print("Nombre d'images corrompues :", images_corrompues.shape[0])
images_corrompues[["nom_fichier", "classe", "chemin"]]

Nombre d'images corrompues : 6


,nom_fichier,classe,chemin
147,cardboard83.jpg,cardboard,../data/raw\cardboard\cardboard83.jpg
326,glass74.jpg,glass,../data/raw\glass\glass74.jpg
446,metal48.jpg,metal,../data/raw\metal\metal48.jpg
633,paper213.jpg,paper,../data/raw\paper\paper213.jpg
791,plastic13.jpg,plastic,../data/raw\plastic\plastic13.jpg
1004,trash3.jpg,trash,../data/raw\trash\trash3.jpg


Partie 3 – Détecter les images vides 
Écrire et se servir d’une fonction qui détecte les images vides : image entièrement noire, image 
entièrement blanche ou image dont les pixels présentent très peu de variation. 

In [7]:
# Partie 3 : fonction qui detecte une image "vide" (noire, blanche, ou tres peu de variation)
def est_vide(chemin_fichier, seuil_ecart_type=5.0):
    """
    Retourne True si l'image est consideree comme vide :
    entierement noire, entierement blanche, ou ecart-type des pixels tres faible.
    seuil_ecart_type : en dessous de cette valeur, on considere qu'il n'y a presque pas de variation.
    """
    try:
        with Image.open(chemin_fichier) as img:
            img_array = np.array(img)
        moyenne = img_array.mean()
        ecart_type = img_array.std()
        # Noire : moyenne proche de 0 | Blanche : moyenne proche de la valeur max (255) | Peu de variation : ecart-type faible
        return ecart_type < seuil_ecart_type
    except Exception:
        return False  # une image corrompue n'est pas traitee ici, deja geree en Partie 2

In [8]:
# Application de la fonction sur les images non corrompues (calcul deja fait via ecart_type_pixels)
df_audit["quasi_vide"] = df_audit["ecart_type_pixels"] < 5.0
images_vides = df_audit[(df_audit["quasi_vide"] == True) & (df_audit["corrompue"] == False)]
print("Nombre d'images quasi vides :", images_vides.shape[0])
images_vides[["nom_fichier", "classe", "ecart_type_pixels"]]

Nombre d'images quasi vides : 2


,nom_fichier,classe,ecart_type_pixels
167,image-blanche-512x384.jpg,cardboard,1.572536
357,image-blanche-512x384.jpg,metal,1.572536


Partie 4 – Détecter les différences de résolution 

In [9]:
# Partie 4, point 1 : analyse des resolutions (largeur x hauteur)
# On travaille uniquement sur les images non corrompues (largeur/hauteur = NaN sinon)
df_valides = df_audit[df_audit["corrompue"] == False].copy()
df_valides["resolution"] = df_valides["largeur"].astype(int).astype(str) + "x" + df_valides["hauteur"].astype(int).astype(str)

# Nombre d'images par resolution, triees de la plus frequente a la moins frequente
frequence_resolutions = df_valides["resolution"].value_counts()
print("Nombre de resolutions distinctes :", frequence_resolutions.shape[0])
print()
print("Resolutions les plus frequentes :")
print(frequence_resolutions.head(10))

Nombre de resolutions distinctes : 4

Resolutions les plus frequentes :
resolution
512x384    1013
32x32         5
48x32         4
40x40         4
Name: count, dtype: int64


In [10]:
# Resolution minimale et maximale, en se basant sur le nombre total de pixels (largeur x hauteur)
df_valides["nb_pixels"] = df_valides["largeur"] * df_valides["hauteur"]

ligne_min = df_valides.loc[df_valides["nb_pixels"].idxmin()]
ligne_max = df_valides.loc[df_valides["nb_pixels"].idxmax()]

print(f"Resolution minimale : {int(ligne_min['largeur'])}x{int(ligne_min['hauteur'])} ({ligne_min['nom_fichier']})")
print(f"Resolution maximale : {int(ligne_max['largeur'])}x{int(ligne_max['hauteur'])} ({ligne_max['nom_fichier']})")

Resolution minimale : 32x32 (cardboard22.jpg)
Resolution maximale : 512x384 (cardboard1.jpg)


In [11]:
# Partie 4, point 2 : identifier les images sous le seuil minimum de 64x64 pixels
SEUIL_MIN_LARGEUR = 64
SEUIL_MIN_HAUTEUR = 64

images_trop_petites = df_valides[
    (df_valides["largeur"] < SEUIL_MIN_LARGEUR) | (df_valides["hauteur"] < SEUIL_MIN_HAUTEUR)
]

print("Nombre d'images trop petites (< 64x64) :", images_trop_petites.shape[0])
images_trop_petites[["nom_fichier", "classe", "largeur", "hauteur"]]

Nombre d'images trop petites (< 64x64) : 13


,nom_fichier,classe,largeur,hauteur
20,cardboard117.jpg,cardboard,48.0,32.0
77,cardboard22.jpg,cardboard,32.0,32.0
133,cardboard70.jpg,cardboard,40.0,40.0
171,glass100.jpg,glass,40.0,40.0
229,glass15.jpg,glass,48.0,32.0
268,glass21.jpg,glass,32.0,32.0
270,glass23.jpg,glass,32.0,32.0
383,metal121.jpg,metal,48.0,32.0
413,metal2.jpg,metal,32.0,32.0
421,metal26.jpg,metal,40.0,40.0


Partie 5 – Détecter les différents canaux 
Déterminer le nombre d'images selon leur nombre de canaux.

In [12]:
# Partie 5 : nombre d'images selon leur nombre de canaux
frequence_canaux = df_valides["nb_canaux"].value_counts().sort_index()
print("Repartition des images par nombre de canaux :")
print(frequence_canaux)
print()

# Detail par mode (grayscale, RGB, RGBA...) pour plus de precision
frequence_modes = df_valides["mode"].value_counts()
print("Repartition des images par mode :")
print(frequence_modes)

Repartition des images par nombre de canaux :
nb_canaux
1.0       2
3.0    1006
4.0      18
Name: count, dtype: int64

Repartition des images par mode :
mode
RGB     1006
RGBA      18
P          2
Name: count, dtype: int64
